# Phase 2: Fine-Tuning Llama 3 8B with Unsloth (Colab Edition)

This notebook takes the synthetic legal contract dataset generated locally and uses it to fine-tune a quantization-ready Small Language Model (SLM). We use Unsloth for highly optimized memory usage and faster training on a free Colab T4 GPU.

### Step 1: Install Dependencies
We use `%pip` to install the official Colab-optimized version of Unsloth and its dependencies.

In [ ]:
%%capture
%pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

: 

### Step 2: Load the Base Model
We load Llama-3.2 3B in 4-bit precision so it fits easily within the 16GB VRAM of a free Google Colab T4 GPU.

In [ ]:
import os
import torch
from unsloth import FastLanguageModel

# Set to 2048 to prevent truncating long legal JSON responses
max_seq_length = 2048 
dtype = None # Auto-detects float16 or bfloat16 based on your hardware
load_in_4bit = True # CRITICAL for memory savings

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### Step 3: Apply LoRA Adapters
We use a lower rank (`r=8`) to prevent overfitting on our small dataset, while targeting both attention (`q_proj`, `v_proj`, etc.) and MLP layers (`gate_proj`, `up_proj`, `down_proj`) to ensure the model learns *how* to process the extraction, not just *where* to attend.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,           # Lowered to 8 to prevent overfitting to synthetic phrasing
    lora_alpha = 16, # 2:1 alpha-to-rank ratio is standard
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,
)

### Step 4: Load, Format, and Split the Dataset
Here we format the dataset using ChatML and split 10% of the data into an `eval_dataset`. This allows us to track `eval_loss` and ensure the model generalizes to unseen contracts.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

dataset_path = "training_dataset.jsonl"

if not os.path.exists(dataset_path):
    print(f"❌ Error: Could not find dataset at '{dataset_path}'")
    print("Please click the Folder icon on the left and upload your JSONL file!")
else:
    raw_dataset = load_dataset("json", data_files=dataset_path, split="train")

    tokenizer = get_chat_template(
        tokenizer,
        chat_template = "llama-3",
        mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"}
    )

    def formatting_prompts_func(examples):
        convos = examples["messages"]
        texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
        return { "text" : texts }

    formatted_dataset = raw_dataset.map(formatting_prompts_func, batched = True)
    
    # SPLIT DATASET: 90% Training, 10% Evaluation
    split_dataset = formatted_dataset.train_test_split(test_size=0.1, seed=3407)
    train_dataset = split_dataset["train"]
    eval_dataset = split_dataset["test"]
    
    print(f"✅ Formatted Dataset. Train: {len(train_dataset)} examples | Eval: {len(eval_dataset)} examples.")

### Step 5: Start Epoch-Based Training with Evaluation
We switched to `num_train_epochs=3` to guarantee the model sees every example multiple times. We also added `eval_strategy` to monitor the loss on our holdout set, preventing silent overfitting.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1, # VRAM constraint tradeoff (compensating with grad accum)
        gradient_accumulation_steps = 8, # Effective batch size = 8
        warmup_ratio = 0.03,             # Replaced warmup_steps for epoch-based scalability
        num_train_epochs = 3,            # Guarantee full dataset passes (Replaced max_steps)
        learning_rate = 1e-4,            # Lowered from 2e-4 to stabilize loss curve
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        eval_strategy = "steps",         # Evaluate on the holdout set
        eval_steps = 25,
        save_strategy = "steps",
        save_steps = 25,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### Step 6: Save the Trained LoRA Adapters & Export GGUF
Once training is done, we save the custom LoRA weights and immediately export the `Q4_K_M` GGUF file for local deployment.

In [ ]:
model.save_pretrained("legal_slm_lora_adapter")
tokenizer.save_pretrained("legal_slm_lora_adapter")
print("✅ Fine-Tuned Adapters saved successfully!")

# Export to 4-bit GGUF for Hugging Face Spaces / llama.cpp inference
model.save_pretrained_gguf("legal_slm", tokenizer, quantization_method="q4_k_m")
print("✅ GGUF Export complete! Download the 'legal_slm-unsloth.Q4_K_M.gguf' file from the panel on the left.")